# 🏆 MedGemma Clinical RAG Pipeline
This notebook demonstrates the end-to-end execution of the MedGemma Sentinel pipeline. It integrates a Vision-Language Model (MedGemma 1.5-4B) with a Retrieval-Augmented Generation (RAG) engine to provide grounded clinical triage and empathetic patient translations.

### 1. Environment Setup & Dependencies
Install required libraries and authenticate with Hugging Face.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets faiss-cpu sentence-transformers textstat evaluate huggingface_hub

import os, torch, zipfile, pandas as pd, numpy as np, textstat, faiss, re
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown
from huggingface_hub import login, whoami

# Output directories
OUTPUT_BASE = "/kaggle/working/output"
os.makedirs(f"{OUTPUT_BASE}/reports", exist_ok=True)
os.makedirs(f"{OUTPUT_BASE}/samples", exist_ok=True)

print("Authenticating...")
HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # Replace with your token if running locally
try:
    if HF_TOKEN != "YOUR_HF_TOKEN_HERE":
        login(token=HF_TOKEN)
        print(f"✅ Authenticated as: {whoami().get('name')}")
    else:
        print("⚠️ HF_TOKEN not set. Gated models will fail to load.")
except Exception as e:
    print(f"❌ Auth Error: {e}")

### 2. Load MedGemma 1.5-4B (4-bit QLoRA)
Load the model with 4-bit quantization to fit within consumer GPU constraints.

In [ ]:
MODEL_ID = "google/medgemma-1.5-4b-it"
print(f"Loading {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, 
    quantization_config=bnb_config, 
    device_map="auto", 
    token=HF_TOKEN, 
    trust_remote_code=True
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, 
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
print("✅ Sentinel Model Ready.")

### 3. Build Medical RAG Engine
Initialize FAISS vector search over the MedQuAD knowledge base.

In [ ]:
print("Setting up RAG Engine...")

# Dynamic path discovery for Kaggle environments
kb_path = next((os.path.join(d, f) for d, _, fs in os.walk("/kaggle/input") for f in fs if "medquad" in d.lower() and (f.endswith(".json") or f.endswith(".csv"))), None)
index, knowledge_base, embedder = None, [], None

try:
    if kb_path:
        if kb_path.endswith(".json"):
            knowledge_base = pd.read_json(kb_path)["answer"].dropna().tolist()
        else:
            df_kb = pd.read_csv(kb_path)
            col = next((c for f in ["answer", "Answer", "Response"] for c in df_kb.columns if f in c), df_kb.columns[-1])
            knowledge_base = df_kb[col].dropna().tolist()
            
        if knowledge_base:
            embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
            embeddings = embedder.encode(knowledge_base[:1000], batch_size=64, show_progress_bar=False)
            index = faiss.IndexFlatL2(embeddings.shape[1])
            index.add(np.array(embeddings).astype("float32"))
            print(f"✅ RAG Activated: {len(knowledge_base[:1000])} clinical documents indexed.")
except Exception as e:
    print(f"⚠️ RAG disabled: {e}")

def get_medical_context(query):
    if not index:
        return "Standard clinical guidelines apply."
    _, I = index.search(np.array(embedder.encode([query])).astype("float32"), 1)
    return knowledge_base[I[0][0]]

### 4. Core Pipeline Functions
Functions to handle multimodal token injection, clinical triage, and empathetic patient translation.

In [ ]:
def generate(prompt, img=None, tokens=250):
    content = [{"type": "image"}, {"type": "text", "text": prompt}] if img else [{"type": "text", "text": prompt}]
    input_text = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True)
    inputs = processor(images=img, text=input_text, return_tensors="pt").to("cuda")
    
    if img is None and "pixel_values" in inputs: 
        del inputs["pixel_values"]
        
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=tokens)
        
    return processor.batch_decode(out, skip_special_tokens=True)[0].split("model")[-1].strip()

def run_pipeline(image, sample_id):
    # 1. Observation
    r1 = generate("Describe the findings in this image.", img=image, tokens=150)
    
    # 2. RAG & Clinical Triage
    ctx = get_medical_context(r1)
    final_clin = generate(f"Context: '{ctx[:200]}'. Refine this report and triage as NORMAL/ABNORMAL.\nInitial: {r1}", img=image, tokens=200)
    
    # 3. Patient Translation
    patient_sum = generate(f"Translate for a patient with empathy and simple terms:\nReport: {final_clin}", tokens=300)
    
    # 4. Metrics & Display
    grade = textstat.flesch_kincaid_grade(patient_sum)
    improvement = 15.3 - grade
    
    display(Markdown(f"---\n### 🔍 Analysis: {sample_id}"))
    display(image.resize((450, 450)))
    display(Markdown(f"#### 🩺 CLINICAL REPORT\n{final_clin}\n\n#### 🤝 PATIENT SUMMARY\n{patient_sum}\n\n**📈 Complexity Reduction:** ~{improvement:.1f} Grade Points (Expert 15.3 → Patient {grade:.1f})"))
    
    # Save Artifacts
    image.save(f"{OUTPUT_BASE}/samples/{sample_id}.png")
    with open(f"{OUTPUT_BASE}/reports/{sample_id}.md", "w") as f:
        f.write(f"# {sample_id}\n{final_clin}\n\n{patient_sum}")

### 5. Execution
Process samples from the VQA-RAD dataset and package the results.

In [ ]:
print("--- Starting Execution ---")
all_files = [os.path.join(d, f) for d, _, fs in os.walk("/kaggle/input") for f in fs]
vqa_json = next((f for f in all_files if "vqa" in f.lower() and f.endswith(".json")), None)

if vqa_json:
    df = pd.read_json(vqa_json).head(2)
    for _, row in df.iterrows():
        fpath = next((f for f in all_files if f.endswith(row["image_name"])), None)
        if fpath:
            img = Image.open(fpath).convert("RGB")
            run_pipeline(img, str(row["image_name"]).split(".")[0])

print("\n--- Packaging Results ---")
with zipfile.ZipFile("/kaggle/working/clinical_rag_pipeline_results.zip", "w") as zf:
    for root, _, files in os.walk(OUTPUT_BASE):
        for f in files:
            zf.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), OUTPUT_BASE))
print("✅ Complete.")